
# 01. Dataset enriquecido de modelagem — contratos e snapshots

Este notebook reconstrói o dataset de modelagem usando como fonte mestre:

`vw_modelagem_contratos_filtros.csv`

A fonte contém os contratos já filtrados para o escopo de obras e inclui atributos adicionais de contrato.

## Princípios

- a nova view define o universo elegível;
- `data_recebimento_provisorio` é a conclusão definitiva;
- o prazo original é `data_ordem_servico + prazo_total`;
- o prazo vigente em cada snapshot considera apenas aditivos assinados até `data_corte`;
- o prazo final do target considera somente aditivos assinados até a conclusão definitiva;
- as fontes auxiliares são filtradas pelos IDs da nova view;
- informações finais ou posteriores à execução não entram como features;
- o dataset é mensal, com uma linha por `id_contrato + data_corte`;
- pesos são calculados por contrato;
- split temporal é definido no nível do contrato.

## Fontes auxiliares

- `vw_bi_parcelas.csv`
- `vw_tparcela_liberacao.csv`
- `vw_taditivovalor.csv`
- `vw_taditivoprazo.csv`

## Target

`target_dias_atraso = max(0, data_recebimento_provisorio - data_termino_vigente_final)`

`target_atraso = 1` quando `target_dias_atraso > 0`.


In [1]:

from pathlib import Path
import hashlib
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

FILES = {
    "contratos": "vw_modelagem_contratos_filtros.csv",
    "parcelas": "vw_bi_parcelas.csv",
    "liberacoes": "vw_tparcela_liberacao.csv",
    "aditivos_valor": "vw_taditivovalor.csv",
    "aditivos_prazo": "vw_taditivoprazo.csv",
}


## 1. Pastas e localização dos arquivos

In [2]:

def find_project_root() -> Path:
    current = Path.cwd().resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "data" / "raw").exists():
            return candidate

    for candidate in [current, *current.parents]:
        if (candidate / "data" / "processed").exists():
            return candidate

    return current


def writable_directory(
    preferred: Path,
    fallback: Path,
) -> Path:
    """
    Usa a pasta padrão do projeto quando ela é gravável.
    Em ambientes restritos, usa uma pasta alternativa.
    """
    for candidate in [preferred, fallback]:
        try:
            candidate.mkdir(
                parents=True,
                exist_ok=True,
            )

            probe = candidate / ".write_probe"

            with probe.open(
                "w",
                encoding="utf-8",
            ) as file:
                file.write("ok")

            probe.unlink()

            return candidate

        except (PermissionError, OSError):
            continue

    raise PermissionError(
        "Nenhuma pasta de saída disponível para gravação."
    )


ROOT = find_project_root()
RAW = ROOT / "data" / "raw"

PROCESSED = writable_directory(
    ROOT / "data" / "processed",
    Path("/mnt/data/modelagem_outputs/processed"),
)

QUALITY = writable_directory(
    ROOT / "reports" / "quality",
    Path("/mnt/data/modelagem_outputs/quality"),
)


def find_input_file(filename: str) -> Path:
    candidates = [
        RAW / filename,
        ROOT / filename,
        Path.cwd() / filename,
        Path("/mnt/data") / filename,
    ]

    for path in candidates:
        if path.exists():
            return path.resolve()

    raise FileNotFoundError(
        f"Arquivo não encontrado: {filename}\n"
        + "\n".join(str(p) for p in candidates)
    )


SOURCE_PATHS = {
    key: find_input_file(filename)
    for key, filename in FILES.items()
}

print("ROOT:", ROOT.resolve())
print("PROCESSED:", PROCESSED.resolve())
print("QUALITY:", QUALITY.resolve())
print()

for key, path in SOURCE_PATHS.items():
    print(f"{key:18s}: {path}")


ROOT: C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras
PROCESSED: C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\data\processed
QUALITY: C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\reports\quality

contratos         : C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\data\raw\vw_modelagem_contratos_filtros.csv
parcelas          : C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\data\raw\vw_bi_parcelas.csv
liberacoes        : C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\data\raw\vw_tparcela_liberacao.csv
aditivos_valor    : C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\data\raw\vw_taditivovalor.csv
aditivos_prazo    : C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\data\raw\vw_taditivoprazo.csv


## 2. Carga e manifesto

In [3]:

def read_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(
        path,
        low_memory=False,
        encoding="utf-8-sig",
    )


def sha256(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


contratos = read_csv(SOURCE_PATHS["contratos"])
parcelas = read_csv(SOURCE_PATHS["parcelas"])
liberacoes = read_csv(SOURCE_PATHS["liberacoes"])
aditivos_valor = read_csv(SOURCE_PATHS["aditivos_valor"])
aditivos_prazo = read_csv(SOURCE_PATHS["aditivos_prazo"])

datasets_raw = {
    "contratos": contratos,
    "parcelas": parcelas,
    "liberacoes": liberacoes,
    "aditivos_valor": aditivos_valor,
    "aditivos_prazo": aditivos_prazo,
}

manifest = pd.DataFrame([
    {
        "fonte": key,
        "arquivo": SOURCE_PATHS[key].name,
        "linhas": len(data),
        "colunas": data.shape[1],
        "sha256": sha256(SOURCE_PATHS[key]),
    }
    for key, data in datasets_raw.items()
])

display(manifest)

manifest.to_csv(
    QUALITY / "manifesto_dataset_enriquecido.csv",
    index=False,
    encoding="utf-8-sig",
)


,fonte,arquivo,linhas,colunas,sha256
0,contratos,vw_modelagem_contratos_filtros.csv,48,50,fb9a1b9bcc1f5657544e4817ac07334617741e479c92e7...
1,parcelas,vw_bi_parcelas.csv,13010,11,33ce2ca7b1f53f4d95428b431c46ca9c7f88a062f33867...
2,liberacoes,vw_tparcela_liberacao.csv,11918,8,b77ba8cc895957e852ef543b85697fbff3912b59564ba2...
3,aditivos_valor,vw_taditivovalor.csv,271,10,384a5614ec6766368935af3e761c773e52f70d8d147ade...
4,aditivos_prazo,vw_taditivoprazo.csv,245,10,338ec4b4799c694dbcdcb5f109add0173eba3f3b499ec5...



## 3. Tipos, datas e validação da fonte mestre

A fonte mestre deve conter uma linha por contrato e campos suficientes para construir o target.

Campos pós-conclusão, como TRP/TRD e status final, são preservados somente na fonte bruta e **não são features**.


In [4]:

def to_dates(df: pd.DataFrame, columns) -> pd.DataFrame:
    result = df.copy()

    for column in columns:
        if column not in result.columns:
            continue

        result[column] = pd.to_datetime(
            result[column],
            errors="coerce",
        )

        valid = result[column].notna()

        result.loc[
            valid & (result[column].dt.year <= 1900),
            column,
        ] = pd.NaT

    return result


def to_numbers(df: pd.DataFrame, columns) -> pd.DataFrame:
    result = df.copy()

    for column in columns:
        if column in result.columns:
            result[column] = pd.to_numeric(
                result[column],
                errors="coerce",
            )

    return result


required_contract_columns = [
    "idContrato",
    "id_setor_divisao",
    "Classificacao_contrato",
    "id_predio",
    "id_fornecedor",
    "ano_contrato",
    "data_assinatura",
    "prazo_total",
    "valor_total_contrato",
    "data_ordem_servico",
    "data_recebimento_provisorio",
    "Valor_licitado",
    "Perc_desconto",
    "Status_contrato",
    "CT_concluido",
    "CT_rescindido",
]

missing = [
    c for c in required_contract_columns
    if c not in contratos.columns
]

if missing:
    raise ValueError(
        f"Colunas obrigatórias ausentes na fonte mestre: {missing}"
    )


contract_numeric = [
    "idContrato",
    "id_setor_divisao",
    "id_classificacao_contrato",
    "id_predio",
    "id_projeto_associado",
    "id_Licitacao",
    "id_fornecedor",
    "numero_contrato",
    "ano_contrato",
    "prazo_total",
    "valor_total_contrato",
    "Valor_licitado",
    "Perc_desconto",
    "Flag_ativo_emissao_OS",
    "Numero_etapas_projeto",
]

contratos = to_numbers(
    contratos,
    contract_numeric,
)

contract_dates = [
    "data_assinatura",
    "data_proposta",
    "DataOrcamento",
    "data_ordem_servico",
    "data_ref_reajuste",
    "data_conclusao",
    "data_recebimento_provisorio",
    "data_assinatura_TRP",
    "data_recebimento_TRD",
    "data_assinatura_TRD",
    "Data_paralisacao",
    "Data_previsao_retomada",
]

contratos = to_dates(
    contratos,
    contract_dates,
)

contratos["Classificacao_contrato"] = (
    contratos["Classificacao_contrato"]
    .astype("string")
    .str.strip()
)

if contratos["idContrato"].isna().any():
    raise ValueError("Há contrato sem idContrato.")

if contratos["idContrato"].duplicated().any():
    duplicados = contratos.loc[
        contratos["idContrato"].duplicated(keep=False),
        ["idContrato", "NumeroContrato"],
    ]

    display(duplicados)

    raise ValueError(
        "A fonte mestre deve ter uma única linha por contrato."
    )

contratos["idContrato"] = contratos["idContrato"].astype(int)

if contratos["data_ordem_servico"].isna().any():
    raise ValueError(
        "Há contratos sem data_ordem_servico válida."
    )

if contratos["data_recebimento_provisorio"].isna().any():
    raise ValueError(
        "Há contratos sem data_recebimento_provisorio válida."
    )

if contratos["prazo_total"].isna().any():
    raise ValueError(
        "Há contratos sem prazo_total válido."
    )

if (contratos["prazo_total"] <= 0).any():
    raise ValueError(
        "Há contratos com prazo_total <= 0."
    )

if (
    contratos["data_recebimento_provisorio"]
    < contratos["data_ordem_servico"]
).any():
    raise ValueError(
        "Há recebimento provisório anterior à ordem de serviço."
    )

print("Contratos na fonte mestre:", len(contratos))
print("IDs únicos:", contratos["idContrato"].nunique())
print()
print("Tipos de contrato:")
display(
    contratos["tipo_de_contrato"]
    .value_counts(dropna=False)
    .rename("contratos")
    .to_frame()
)
print("Classificações:")
display(
    contratos["Classificacao_contrato"]
    .value_counts(dropna=False)
    .rename("contratos")
    .to_frame()
)


Contratos na fonte mestre: 48
IDs únicos: 48

Tipos de contrato:


,contratos
tipo_de_contrato,
Obra,48


Classificações:


,contratos
Classificacao_contrato,
Reforma Parcial,38
Construção,6
Reforma Parcial e Ampliação,4


## 4. Tratamento e filtro das fontes auxiliares

In [5]:

parcelas = to_numbers(
    parcelas,
    [
        "id_parcela",
        "id_contrato",
        "valor_reajustado",
        "ordem_status",
        "valor_ja_liberado",
    ],
)

parcelas = to_dates(
    parcelas,
    [
        "data_prevista_parcela",
        "data_efetiva",
        "data_liberacao",
    ],
)

liberacoes = to_numbers(
    liberacoes,
    [
        "id_Contrato",
        "id_Parcela",
        "Abono_atraso",
        "Atraso_considerado",
        "Valor_total_liberado",
    ],
)

liberacoes = to_dates(
    liberacoes,
    [
        "Data_efet_conclusao",
        "Data_liberação",
    ],
)

aditivos_valor = to_numbers(
    aditivos_valor,
    ["idContrato"],
)

aditivos_valor = to_dates(
    aditivos_valor,
    ["Data_assinatura_TA_valor"],
)

aditivos_prazo = to_numbers(
    aditivos_prazo,
    [
        "idContrato",
        "Aditado_em_dias",
    ],
)

aditivos_prazo = to_dates(
    aditivos_prazo,
    [
        "Data_termino_anterior",
        "Data_termino_nova",
        "Data_assinatura_ta_prazo",
    ],
)

ids_modelagem = set(
    contratos["idContrato"].astype(int)
)

parcelas = parcelas[
    parcelas["id_contrato"].isin(ids_modelagem)
].copy()

liberacoes = liberacoes[
    liberacoes["id_Contrato"].isin(ids_modelagem)
].copy()

aditivos_valor = aditivos_valor[
    aditivos_valor["idContrato"].isin(ids_modelagem)
].copy()

aditivos_prazo = aditivos_prazo[
    aditivos_prazo["idContrato"].isin(ids_modelagem)
].copy()

coverage = pd.DataFrame([
    {
        "fonte": "contratos",
        "linhas": len(contratos),
        "contratos": contratos["idContrato"].nunique(),
    },
    {
        "fonte": "parcelas",
        "linhas": len(parcelas),
        "contratos": parcelas["id_contrato"].nunique(),
    },
    {
        "fonte": "liberacoes",
        "linhas": len(liberacoes),
        "contratos": liberacoes["id_Contrato"].nunique(),
    },
    {
        "fonte": "aditivos_valor",
        "linhas": len(aditivos_valor),
        "contratos": aditivos_valor["idContrato"].nunique(),
    },
    {
        "fonte": "aditivos_prazo",
        "linhas": len(aditivos_prazo),
        "contratos": aditivos_prazo["idContrato"].nunique(),
    },
])

display(coverage)


,fonte,linhas,contratos
0,contratos,48,48
1,parcelas,897,47
2,liberacoes,886,47
3,aditivos_valor,134,38
4,aditivos_prazo,85,37



## 5. Prazo final e targets

O prazo final é reconstruído a partir dos aditivos de prazo assinados até `data_recebimento_provisorio`.

A observação de conclusão não altera o target automaticamente. Ela é usada apenas em auditoria, pois representa informação conhecida depois do desfecho.


In [6]:

contratos["data_termino_original"] = (
    contratos["data_ordem_servico"]
    + pd.to_timedelta(
        contratos["prazo_total"],
        unit="D",
    )
)

final_deadline_rows = []

for _, contrato in contratos.iterrows():
    cid = int(contrato["idContrato"])
    conclusao = contrato[
        "data_recebimento_provisorio"
    ]

    ap = aditivos_prazo[
        aditivos_prazo["idContrato"].eq(cid)
        & aditivos_prazo[
            "Data_assinatura_ta_prazo"
        ].notna()
        & aditivos_prazo[
            "Data_termino_nova"
        ].notna()
        & (
            aditivos_prazo[
                "Data_assinatura_ta_prazo"
            ]
            <= conclusao
        )
    ].sort_values(
        [
            "Data_assinatura_ta_prazo",
            "Data_termino_nova",
        ]
    )

    final_deadline = (
        ap["Data_termino_nova"].iloc[-1]
        if len(ap)
        else contrato["data_termino_original"]
    )

    final_deadline_rows.append({
        "idContrato": cid,
        "data_termino_vigente_final": final_deadline,
        "qtd_aditivos_prazo_ate_conclusao": len(ap),
    })

final_deadlines = pd.DataFrame(
    final_deadline_rows
)

contratos = contratos.merge(
    final_deadlines,
    how="left",
    on="idContrato",
    validate="one_to_one",
)

contratos["target_dias_atraso"] = (
    contratos["data_recebimento_provisorio"]
    - contratos["data_termino_vigente_final"]
).dt.days.clip(lower=0)

contratos["target_atraso"] = (
    contratos["target_dias_atraso"] > 0
).astype("int8")

contratos["diferenca_vigente_original_dias"] = (
    contratos["data_termino_vigente_final"]
    - contratos["data_termino_original"]
).dt.days

print(
    "Contratos com atraso:",
    int(contratos["target_atraso"].sum()),
)
print(
    "Contratos sem atraso:",
    int((contratos["target_atraso"] == 0).sum()),
)
print(
    "Taxa de atraso:",
    f"{contratos['target_atraso'].mean():.1%}",
)
print(
    "Média target_dias_atraso:",
    round(contratos["target_dias_atraso"].mean(), 2),
)
print(
    "Mediana target_dias_atraso:",
    round(contratos["target_dias_atraso"].median(), 2),
)

display(
    contratos[
        [
            "idContrato",
            "NumeroContrato",
            "data_ordem_servico",
            "data_termino_original",
            "data_termino_vigente_final",
            "data_recebimento_provisorio",
            "target_dias_atraso",
            "target_atraso",
        ]
    ].sort_values(
        "target_dias_atraso",
        ascending=False,
    ).head(10)
)


Contratos com atraso: 20
Contratos sem atraso: 28
Taxa de atraso: 41.7%
Média target_dias_atraso: 32.33
Mediana target_dias_atraso: 0.0


,idContrato,NumeroContrato,data_ordem_servico,data_termino_original,data_termino_vigente_final,data_recebimento_provisorio,target_dias_atraso,target_atraso
12,51,118/2019,2019-07-10,2019-09-08,2019-09-08,2020-09-22,380,1
18,59,6/2020,2020-10-14,2021-12-08,2022-10-20,2023-07-28,281,1
47,377,20/2025,2025-04-29,2025-07-28,2025-09-11,2026-02-10,152,1
1,12,32/2018,2018-03-22,2018-06-20,2018-06-20,2018-11-06,139,1
3,29,210/2018,2019-01-29,2019-09-26,2020-03-22,2020-07-06,106,1
19,61,23/2020,2020-08-11,2021-06-07,2022-02-04,2022-05-12,97,1
31,205,3/2023,2023-03-06,2024-10-26,2024-12-25,2025-03-10,75,1
30,202,50/2022,2023-01-30,2023-04-30,2023-05-30,2023-07-09,40,1
28,195,22/2022,2022-08-15,2023-05-12,2023-07-11,2023-08-18,38,1
38,272,6/2024,2024-03-21,2024-05-20,2024-05-20,2024-06-19,30,1



## 6. Features enriquecidas do contrato

São criadas apenas variáveis disponíveis antes ou no início da execução.

### Usadas como candidatas

- classificação;
- setor/divisão;
- prazo original;
- valor total do contrato;
- valor licitado quando informado;
- percentual de desconto quando informado;
- relação valor do contrato / valor licitado;
- intervalo assinatura → OS;
- intervalo proposta → OS;
- intervalo orçamento → OS;
- intervalo referência de reajuste → OS;
- experiência histórica do fornecedor antes da OS;
- experiência histórica do prédio antes da OS.

### Não usadas diretamente

IDs de fornecedor, prédio, licitação e projeto não entram como categorias de alta cardinalidade. Eles servem apenas para construir históricos point-in-time.


In [7]:

# ---------------------------------------------------------
# Valores de licitação
# ---------------------------------------------------------
contratos["flag_valor_licitado_informado"] = (
    contratos["Valor_licitado"].fillna(0) > 0
).astype("int8")

contratos["valor_licitado_modelo"] = (
    contratos["Valor_licitado"]
    .where(
        contratos[
            "flag_valor_licitado_informado"
        ].eq(1)
    )
)

contratos["perc_desconto_modelo"] = (
    contratos["Perc_desconto"]
    .where(
        contratos[
            "flag_valor_licitado_informado"
        ].eq(1)
    )
)

contratos[
    "razao_valor_contrato_valor_licitado"
] = np.where(
    contratos["valor_licitado_modelo"].gt(0),
    contratos["valor_total_contrato"]
    / contratos["valor_licitado_modelo"],
    np.nan,
)

contratos["log_valor_total_contrato"] = np.log1p(
    contratos["valor_total_contrato"].clip(lower=0)
)

contratos["log_valor_licitado"] = np.log1p(
    contratos["valor_licitado_modelo"].clip(lower=0)
)

# ---------------------------------------------------------
# Intervalos pré-OS
# ---------------------------------------------------------
pre_os_dates = {
    "data_assinatura": "dias_assinatura_ate_os",
    "data_proposta": "dias_proposta_ate_os",
    "DataOrcamento": "dias_orcamento_ate_os",
    "data_ref_reajuste": "dias_ref_reajuste_ate_os",
}

for source_column, feature_column in pre_os_dates.items():
    delta = (
        contratos["data_ordem_servico"]
        - contratos[source_column]
    ).dt.days

    # Datas posteriores à OS não seriam point-in-time.
    contratos[feature_column] = delta.where(
        delta >= 0
    )

# ---------------------------------------------------------
# Histórico do fornecedor/prédio conhecido antes de cada OS
# ---------------------------------------------------------
history_rows = []

for _, contrato in contratos.iterrows():
    cid = int(contrato["idContrato"])
    data_os = contrato["data_ordem_servico"]

    anteriores_inicio = contratos[
        contratos["data_ordem_servico"] < data_os
    ]

    fornecedor_anteriores = anteriores_inicio[
        anteriores_inicio["id_fornecedor"].eq(
            contrato["id_fornecedor"]
        )
    ]

    predio_anteriores = anteriores_inicio[
        anteriores_inicio["id_predio"].eq(
            contrato["id_predio"]
        )
    ]

    # Resultado histórico só pode ser usado se o contrato anterior
    # já estava concluído antes da OS atual.
    fornecedor_concluidos = contratos[
        contratos["id_fornecedor"].eq(
            contrato["id_fornecedor"]
        )
        & (
            contratos[
                "data_recebimento_provisorio"
            ]
            < data_os
        )
    ]

    predio_concluidos = contratos[
        contratos["id_predio"].eq(
            contrato["id_predio"]
        )
        & (
            contratos[
                "data_recebimento_provisorio"
            ]
            < data_os
        )
    ]

    history_rows.append({
        "idContrato": cid,

        "fornecedor_contratos_anteriores": (
            len(fornecedor_anteriores)
        ),
        "fornecedor_valor_anterior_acumulado": (
            fornecedor_anteriores[
                "valor_total_contrato"
            ].sum()
            if len(fornecedor_anteriores)
            else 0.0
        ),
        "fornecedor_contratos_concluidos_antes_os": (
            len(fornecedor_concluidos)
        ),
        "fornecedor_taxa_atraso_historica": (
            fornecedor_concluidos[
                "target_atraso"
            ].mean()
            if len(fornecedor_concluidos)
            else np.nan
        ),
        "fornecedor_atraso_medio_dias_historico": (
            fornecedor_concluidos[
                "target_dias_atraso"
            ].mean()
            if len(fornecedor_concluidos)
            else np.nan
        ),

        "predio_contratos_anteriores": (
            len(predio_anteriores)
        ),
        "predio_contratos_concluidos_antes_os": (
            len(predio_concluidos)
        ),
        "predio_taxa_atraso_historica": (
            predio_concluidos[
                "target_atraso"
            ].mean()
            if len(predio_concluidos)
            else np.nan
        ),
        "predio_atraso_medio_dias_historico": (
            predio_concluidos[
                "target_dias_atraso"
            ].mean()
            if len(predio_concluidos)
            else np.nan
        ),
    })

history = pd.DataFrame(history_rows)

contratos = contratos.merge(
    history,
    how="left",
    on="idContrato",
    validate="one_to_one",
)

contratos[
    "fornecedor_primeiro_contrato"
] = (
    contratos[
        "fornecedor_contratos_anteriores"
    ].eq(0)
).astype("int8")

contratos[
    "predio_primeiro_contrato"
] = (
    contratos[
        "predio_contratos_anteriores"
    ].eq(0)
).astype("int8")

print("Features enriquecidas criadas.")


Features enriquecidas criadas.


## 7. Auditoria de possíveis exceções semânticas

In [8]:

# Observações finais NÃO entram como features.
# Apenas sinalizamos registros que merecem revisão humana.

obs = (
    contratos["observacao_conclusao"]
    .fillna("")
    .astype(str)
    .str.lower()
)

mask_revisao = (
    obs.str.contains(
        "sem atraso|abon|considerada sem atraso",
        regex=True,
    )
    & contratos["target_atraso"].eq(1)
)

target_review = contratos.loc[
    mask_revisao,
    [
        "idContrato",
        "NumeroContrato",
        "data_termino_vigente_final",
        "data_recebimento_provisorio",
        "target_dias_atraso",
        "observacao_conclusao",
    ],
].copy()

target_review.to_csv(
    QUALITY / "contratos_revisao_semantica_target.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    "Contratos com possível divergência entre atraso cronológico "
    "e observação textual:",
    len(target_review),
)

if len(target_review):
    display(target_review)


Contratos com possível divergência entre atraso cronológico e observação textual: 1


,idContrato,NumeroContrato,data_termino_vigente_final,data_recebimento_provisorio,target_dias_atraso,observacao_conclusao
3,29,210/2018,2020-03-22,2020-07-06,106,Considerada sem atraso: Cabe ressaltar que as ...


## 8. Calendário mensal de snapshots

In [9]:

calendar_rows = []

for _, contrato in contratos.iterrows():
    cid = int(contrato["idContrato"])
    inicio = contrato["data_ordem_servico"]
    fim = contrato[
        "data_recebimento_provisorio"
    ]

    cortes = list(
        pd.date_range(
            inicio,
            fim,
            freq="ME",
        )
    )

    if not cortes or cortes[0] > inicio:
        cortes = [inicio] + cortes

    if cortes[-1] != fim:
        cortes.append(fim)

    for data_corte in sorted(set(cortes)):
        calendar_rows.append(
            (cid, pd.Timestamp(data_corte))
        )

calendar = pd.DataFrame(
    calendar_rows,
    columns=[
        "id_contrato",
        "data_corte",
    ],
)

if calendar.duplicated(
    [
        "id_contrato",
        "data_corte",
    ]
).any():
    raise ValueError(
        "Há snapshots duplicados."
    )

print("Snapshots:", len(calendar))
print(
    "Contratos:",
    calendar["id_contrato"].nunique(),
)
print(
    "Período:",
    calendar["data_corte"].min(),
    "a",
    calendar["data_corte"].max(),
)


Snapshots: 633
Contratos: 48
Período: 2018-03-22 00:00:00 a 2026-02-10 00:00:00



## 9. Construção das features point-in-time

Parcelas, liberações e aditivos usam apenas fatos conhecidos em `data_corte`.

O prazo final usado no target permanece isolado em coluna de controle.


In [10]:

contratos_idx = contratos.set_index(
    "idContrato",
    drop=False,
)

parcelas_por_contrato = {
    int(k): g.copy()
    for k, g in parcelas.groupby("id_contrato")
    if pd.notna(k)
}

liberacoes_por_contrato = {
    int(k): g.copy()
    for k, g in liberacoes.groupby("id_Contrato")
    if pd.notna(k)
}

aditivos_valor_por_contrato = {
    int(k): g.copy()
    for k, g in aditivos_valor.groupby("idContrato")
    if pd.notna(k)
}

aditivos_prazo_por_contrato = {
    int(k): g.copy()
    for k, g in aditivos_prazo.groupby("idContrato")
    if pd.notna(k)
}

empty_parcelas = parcelas.iloc[0:0].copy()
empty_liberacoes = liberacoes.iloc[0:0].copy()
empty_aditivos_valor = aditivos_valor.iloc[0:0].copy()
empty_aditivos_prazo = aditivos_prazo.iloc[0:0].copy()


def parcel_window_features(
    concluidas: pd.DataFrame,
    data_corte: pd.Timestamp,
    days: int,
) -> dict:
    window = concluidas[
        concluidas["data_efetiva"]
        >= data_corte - pd.Timedelta(days=days)
    ]

    delays = (
        window["data_efetiva"]
        - window["data_prevista_parcela"]
    ).dt.days.dropna()

    return {
        f"recent_qtd_parcelas_concluidas_{days}d": (
            len(window)
        ),
        f"recent_qtd_parcelas_atrasadas_{days}d": (
            int((delays > 0).sum())
            if len(delays)
            else 0
        ),
        f"recent_pct_parcelas_atrasadas_{days}d": (
            float((delays > 0).mean())
            if len(delays)
            else np.nan
        ),
        f"recent_atraso_medio_parcelas_{days}d": (
            float(delays.mean())
            if len(delays)
            else np.nan
        ),
        f"recent_atraso_maximo_parcelas_{days}d": (
            float(delays.max())
            if len(delays)
            else np.nan
        ),
    }


def release_window_features(
    released: pd.DataFrame,
    data_corte: pd.Timestamp,
    days: int,
) -> dict:
    window = released[
        released["Data_liberação"]
        >= data_corte - pd.Timedelta(days=days)
    ]

    return {
        f"recent_qtd_liberacoes_{days}d": len(window),
        f"recent_valor_liberado_{days}d": (
            float(
                window[
                    "Valor_total_liberado"
                ].sum(min_count=1)
            )
            if len(window)
            else 0.0
        ),
        f"recent_atraso_medio_liberacao_{days}d": (
            float(
                window[
                    "Atraso_considerado"
                ].mean()
            )
            if len(window)
            else np.nan
        ),
    }


rows = []

for cid, data_corte in calendar.itertuples(
    index=False,
    name=None,
):
    contrato = contratos_idx.loc[cid]

    data_os = contrato["data_ordem_servico"]
    data_termino_original = contrato[
        "data_termino_original"
    ]

    # ---------------------------------------------------------
    # Prazo vigente conhecido na fotografia
    # ---------------------------------------------------------
    ap_all = aditivos_prazo_por_contrato.get(
        cid,
        empty_aditivos_prazo,
    )

    ap = ap_all[
        ap_all["Data_assinatura_ta_prazo"].notna()
        & ap_all["Data_termino_nova"].notna()
        & (
            ap_all["Data_assinatura_ta_prazo"]
            <= data_corte
        )
    ].sort_values(
        [
            "Data_assinatura_ta_prazo",
            "Data_termino_nova",
        ]
    )

    data_termino_vigente_corte = (
        ap["Data_termino_nova"].iloc[-1]
        if len(ap)
        else data_termino_original
    )

    # ---------------------------------------------------------
    # Parcelas
    # ---------------------------------------------------------
    pi = parcelas_por_contrato.get(
        cid,
        empty_parcelas,
    )

    parcelas_previstas = pi[
        pi["data_prevista_parcela"].notna()
        & (
            pi["data_prevista_parcela"]
            <= data_corte
        )
    ]

    parcelas_concluidas = pi[
        pi["data_efetiva"].notna()
        & (
            pi["data_efetiva"]
            <= data_corte
        )
    ]

    atrasos_parcelas = (
        parcelas_concluidas["data_efetiva"]
        - parcelas_concluidas["data_prevista_parcela"]
    ).dt.days.dropna()

    qtd_previstas = len(
        parcelas_previstas
    )

    qtd_concluidas = len(
        parcelas_concluidas
    )

    razao_concluidas_previstas_raw = (
        qtd_concluidas / qtd_previstas
        if qtd_previstas
        else np.nan
    )

    pct_concluidas_vs_previstas = (
        min(
            1.0,
            razao_concluidas_previstas_raw,
        )
        if pd.notna(
            razao_concluidas_previstas_raw
        )
        else np.nan
    )

    saldo_parcelas = (
        qtd_concluidas - qtd_previstas
    )

    # ---------------------------------------------------------
    # Liberações
    # ---------------------------------------------------------
    li_all = liberacoes_por_contrato.get(
        cid,
        empty_liberacoes,
    )

    li = li_all[
        li_all["Data_liberação"].notna()
        & (
            li_all["Data_liberação"]
            <= data_corte
        )
    ]

    valor_liberado_acumulado = (
        float(
            li[
                "Valor_total_liberado"
            ].sum(min_count=1)
        )
        if len(li)
        else 0.0
    )

    pct_financeiro_liberado = (
        valor_liberado_acumulado
        / contrato["valor_total_contrato"]
        if (
            pd.notna(
                contrato[
                    "valor_total_contrato"
                ]
            )
            and contrato[
                "valor_total_contrato"
            ] > 0
        )
        else np.nan
    )

    # ---------------------------------------------------------
    # Aditivos de valor
    # ---------------------------------------------------------
    av_all = aditivos_valor_por_contrato.get(
        cid,
        empty_aditivos_valor,
    )

    av = av_all[
        av_all[
            "Data_assinatura_TA_valor"
        ].notna()
        & (
            av_all[
                "Data_assinatura_TA_valor"
            ]
            <= data_corte
        )
    ]

    # ---------------------------------------------------------
    # Features de prazo
    # ---------------------------------------------------------
    dias_desde_os = (
        data_corte - data_os
    ).days

    dias_restantes_prazo_vigente = (
        data_termino_vigente_corte
        - data_corte
    ).days

    prazo_vigente_total_dias = (
        data_termino_vigente_corte
        - data_os
    ).days

    pct_prazo_consumido = (
        dias_desde_os
        / prazo_vigente_total_dias
        if prazo_vigente_total_dias > 0
        else np.nan
    )

    pct_prazo_restante = (
        dias_restantes_prazo_vigente
        / prazo_vigente_total_dias
        if prazo_vigente_total_dias > 0
        else np.nan
    )

    # ---------------------------------------------------------
    # Features recentes
    # ---------------------------------------------------------
    recent = {}

    for days in [30, 60, 90]:
        recent.update(
            parcel_window_features(
                parcelas_concluidas,
                data_corte,
                days,
            )
        )

        recent.update(
            release_window_features(
                li,
                data_corte,
                days,
            )
        )

    atraso_medio_hist = (
        float(
            atrasos_parcelas.mean()
        )
        if len(atrasos_parcelas)
        else np.nan
    )

    atraso_medio_90d = recent[
        "recent_atraso_medio_parcelas_90d"
    ]

    delta_atraso_90d_vs_historico = (
        atraso_medio_90d
        - atraso_medio_hist
        if (
            pd.notna(
                atraso_medio_90d
            )
            and pd.notna(
                atraso_medio_hist
            )
        )
        else np.nan
    )

    dias_aditados_acumulados = (
        float(
            ap[
                "Aditado_em_dias"
            ].sum(min_count=1)
        )
        if len(ap)
        else 0.0
    )

    pct_dias_aditados = (
        dias_aditados_acumulados
        / contrato["prazo_total"]
        if contrato["prazo_total"] > 0
        else np.nan
    )

    gap_prazo_execucao = (
        pct_prazo_consumido
        - pct_concluidas_vs_previstas
        if (
            pd.notna(
                pct_prazo_consumido
            )
            and pd.notna(
                pct_concluidas_vs_previstas
            )
        )
        else np.nan
    )

    gap_prazo_financeiro = (
        pct_prazo_consumido
        - pct_financeiro_liberado
        if (
            pd.notna(
                pct_prazo_consumido
            )
            and pd.notna(
                pct_financeiro_liberado
            )
        )
        else np.nan
    )

    # ---------------------------------------------------------
    # Linha do snapshot
    # ---------------------------------------------------------
    row = {
        "id_contrato": cid,
        "data_corte": data_corte,

        # atributos estáticos/seguramente pré-OS
        "classificacao_contrato": (
            contrato[
                "Classificacao_contrato"
            ]
        ),
        "id_setor_divisao": (
            contrato["id_setor_divisao"]
        ),
        "ano_contrato": (
            contrato["ano_contrato"]
        ),
        "prazo_original_dias": (
            contrato["prazo_total"]
        ),
        "valor_total_contrato": (
            contrato[
                "valor_total_contrato"
            ]
        ),
        "log_valor_total_contrato": (
            contrato[
                "log_valor_total_contrato"
            ]
        ),
        "flag_valor_licitado_informado": (
            contrato[
                "flag_valor_licitado_informado"
            ]
        ),
        "valor_licitado_modelo": (
            contrato[
                "valor_licitado_modelo"
            ]
        ),
        "log_valor_licitado": (
            contrato[
                "log_valor_licitado"
            ]
        ),
        "perc_desconto_modelo": (
            contrato[
                "perc_desconto_modelo"
            ]
        ),
        "razao_valor_contrato_valor_licitado": (
            contrato[
                "razao_valor_contrato_valor_licitado"
            ]
        ),
        "dias_assinatura_ate_os": (
            contrato[
                "dias_assinatura_ate_os"
            ]
        ),
        "dias_proposta_ate_os": (
            contrato[
                "dias_proposta_ate_os"
            ]
        ),
        "dias_orcamento_ate_os": (
            contrato[
                "dias_orcamento_ate_os"
            ]
        ),
        "dias_ref_reajuste_ate_os": (
            contrato[
                "dias_ref_reajuste_ate_os"
            ]
        ),

        # histórico point-in-time da entidade
        "fornecedor_contratos_anteriores": (
            contrato[
                "fornecedor_contratos_anteriores"
            ]
        ),
        "fornecedor_valor_anterior_acumulado": (
            contrato[
                "fornecedor_valor_anterior_acumulado"
            ]
        ),
        "fornecedor_contratos_concluidos_antes_os": (
            contrato[
                "fornecedor_contratos_concluidos_antes_os"
            ]
        ),
        "fornecedor_taxa_atraso_historica": (
            contrato[
                "fornecedor_taxa_atraso_historica"
            ]
        ),
        "fornecedor_atraso_medio_dias_historico": (
            contrato[
                "fornecedor_atraso_medio_dias_historico"
            ]
        ),
        "fornecedor_primeiro_contrato": (
            contrato[
                "fornecedor_primeiro_contrato"
            ]
        ),
        "predio_contratos_anteriores": (
            contrato[
                "predio_contratos_anteriores"
            ]
        ),
        "predio_contratos_concluidos_antes_os": (
            contrato[
                "predio_contratos_concluidos_antes_os"
            ]
        ),
        "predio_taxa_atraso_historica": (
            contrato[
                "predio_taxa_atraso_historica"
            ]
        ),
        "predio_atraso_medio_dias_historico": (
            contrato[
                "predio_atraso_medio_dias_historico"
            ]
        ),
        "predio_primeiro_contrato": (
            contrato[
                "predio_primeiro_contrato"
            ]
        ),

        # prazo no snapshot
        "data_os": data_os,
        "data_termino_original": (
            data_termino_original
        ),
        "data_termino_vigente_corte": (
            data_termino_vigente_corte
        ),
        "dias_desde_os": dias_desde_os,
        "dias_restantes_prazo_vigente": (
            dias_restantes_prazo_vigente
        ),
        "prazo_vigente_total_dias": (
            prazo_vigente_total_dias
        ),
        "pct_prazo_consumido": (
            pct_prazo_consumido
        ),
        "pct_prazo_restante": (
            pct_prazo_restante
        ),

        # aditivos
        "qtd_aditivos_prazo": len(ap),
        "dias_aditados_acumulados": (
            dias_aditados_acumulados
        ),
        "pct_dias_aditados_sobre_prazo_original": (
            pct_dias_aditados
        ),
        "dias_desde_ultimo_aditivo_prazo": (
            (
                data_corte
                - ap[
                    "Data_assinatura_ta_prazo"
                ].max()
            ).days
            if len(ap)
            else np.nan
        ),
        "qtd_aditivos_valor": len(av),
        "dias_desde_ultimo_aditivo_valor": (
            (
                data_corte
                - av[
                    "Data_assinatura_TA_valor"
                ].max()
            ).days
            if len(av)
            else np.nan
        ),

        # parcelas
        "qtd_parcelas_previstas_ate_corte": (
            qtd_previstas
        ),
        "qtd_parcelas_concluidas_ate_corte": (
            qtd_concluidas
        ),
        "pct_parcelas_concluidas_vs_previstas": (
            pct_concluidas_vs_previstas
        ),
        "razao_parcelas_concluidas_previstas_raw": (
            razao_concluidas_previstas_raw
        ),
        "saldo_parcelas_concluidas_vs_previstas": (
            saldo_parcelas
        ),
        "atraso_medio_parcelas_dias": (
            atraso_medio_hist
        ),
        "atraso_mediano_parcelas_dias": (
            float(
                atrasos_parcelas.median()
            )
            if len(atrasos_parcelas)
            else np.nan
        ),
        "atraso_maximo_parcelas_dias": (
            float(
                atrasos_parcelas.max()
            )
            if len(atrasos_parcelas)
            else np.nan
        ),
        "qtd_parcelas_atrasadas": (
            int(
                (
                    atrasos_parcelas > 0
                ).sum()
            )
            if len(atrasos_parcelas)
            else 0
        ),
        "pct_parcelas_atrasadas": (
            float(
                (
                    atrasos_parcelas > 0
                ).mean()
            )
            if len(atrasos_parcelas)
            else np.nan
        ),
        "dias_desde_ultima_parcela_concluida": (
            (
                data_corte
                - parcelas_concluidas[
                    "data_efetiva"
                ].max()
            ).days
            if len(parcelas_concluidas)
            else np.nan
        ),
        "delta_atraso_90d_vs_historico": (
            delta_atraso_90d_vs_historico
        ),

        # liberações
        "qtd_liberacoes": len(li),
        "valor_liberado_acumulado": (
            valor_liberado_acumulado
        ),
        "pct_financeiro_liberado": (
            pct_financeiro_liberado
        ),
        "atraso_medio_liberacao_dias": (
            float(
                li[
                    "Atraso_considerado"
                ].mean()
            )
            if len(li)
            else np.nan
        ),
        "atraso_maximo_liberacao_dias": (
            float(
                li[
                    "Atraso_considerado"
                ].max()
            )
            if len(li)
            else np.nan
        ),
        "dias_desde_ultima_liberacao": (
            (
                data_corte
                - li[
                    "Data_liberação"
                ].max()
            ).days
            if len(li)
            else np.nan
        ),

        # interações
        "gap_prazo_execucao": (
            gap_prazo_execucao
        ),
        "gap_prazo_financeiro": (
            gap_prazo_financeiro
        ),

        "teve_parcela_concluida": int(
            len(parcelas_concluidas) > 0
        ),
        "teve_liberacao": int(
            len(li) > 0
        ),
        "teve_aditivo_prazo": int(
            len(ap) > 0
        ),
        "teve_aditivo_valor": int(
            len(av) > 0
        ),

        # controle — jamais usar como feature
        "status_final_controle": (
            contrato["Status_contrato"]
        ),
        "data_conclusao_sistema_controle": (
            contrato["data_conclusao"]
        ),
        "data_conclusao_definitiva": (
            contrato[
                "data_recebimento_provisorio"
            ]
        ),
        "data_conclusao_controle": (
            contrato[
                "data_recebimento_provisorio"
            ]
        ),
        "data_termino_vigente_final_controle": (
            contrato[
                "data_termino_vigente_final"
            ]
        ),
        "diferenca_vigente_original_final_controle": (
            contrato[
                "diferenca_vigente_original_dias"
            ]
        ),
        "target_dias_atraso": (
            float(
                contrato[
                    "target_dias_atraso"
                ]
            )
        ),
        "target_atraso": (
            int(
                contrato[
                    "target_atraso"
                ]
            )
        ),
    }

    row.update(recent)

    rows.append(row)


dataset = pd.DataFrame(rows)

dataset["ano_corte"] = (
    dataset["data_corte"].dt.year
)

dataset["mes_corte"] = (
    dataset["data_corte"].dt.month
)

dataset["mes_sin"] = np.sin(
    2
    * np.pi
    * dataset["mes_corte"]
    / 12
)

dataset["mes_cos"] = np.cos(
    2
    * np.pi
    * dataset["mes_corte"]
    / 12
)

print("Dataset construído:", dataset.shape)


Dataset construído: (633, 102)


## 10. Elegibilidade, pesos e split temporal

In [11]:

dataset[
    "faixa_antecedencia"
] = pd.cut(
    dataset[
        "dias_restantes_prazo_vigente"
    ],
    [
        -np.inf,
        29,
        89,
        179,
        np.inf,
    ],
    labels=[
        "<30d",
        "30-89d",
        "90-179d",
        ">=180d",
    ],
)

dataset["flag_apto_treino"] = (
    dataset["data_corte"]
    < dataset[
        "data_conclusao_definitiva"
    ]
).astype("int8")

dataset[
    "flag_apto_classificacao"
] = (
    dataset[
        "flag_apto_treino"
    ].eq(1)
    & dataset[
        "dias_restantes_prazo_vigente"
    ].ge(0)
).astype("int8")


def normalized_equal_contract_weight(
    data: pd.DataFrame,
    eligible_column: str,
) -> pd.Series:
    eligible = data[
        eligible_column
    ].eq(1)

    counts = (
        data.loc[
            eligible
        ]
        .groupby(
            "id_contrato"
        )
        .size()
    )

    raw = data[
        "id_contrato"
    ].map(
        lambda cid: (
            1.0 / counts[cid]
            if cid in counts.index
            else np.nan
        )
    )

    mean_raw = raw[
        eligible
    ].mean()

    return (
        raw / mean_raw
        if pd.notna(mean_raw)
        and mean_raw > 0
        else raw
    )


dataset[
    "sample_weight_contrato"
] = normalized_equal_contract_weight(
    dataset,
    "flag_apto_treino",
)

dataset[
    "sample_weight_classificacao"
] = normalized_equal_contract_weight(
    dataset,
    "flag_apto_classificacao",
)

# ---------------------------------------------------------
# Split temporal por contrato
# ---------------------------------------------------------
contract_order = (
    contratos[
        [
            "idContrato",
            "data_recebimento_provisorio",
        ]
    ]
    .rename(
        columns={
            "idContrato": "id_contrato",
            "data_recebimento_provisorio": (
                "data_conclusao_definitiva"
            ),
        }
    )
    .sort_values(
        [
            "data_conclusao_definitiva",
            "id_contrato",
        ]
    )
    .reset_index(drop=True)
)

n = len(contract_order)

n_train = int(
    np.floor(
        n * 0.70
    )
)

n_validation = int(
    np.floor(
        n * 0.15
    )
)

ids_train = set(
    contract_order.iloc[
        :n_train
    ]["id_contrato"]
)

ids_validation = set(
    contract_order.iloc[
        n_train:
        n_train
        + n_validation
    ]["id_contrato"]
)

ids_test = set(
    contract_order.iloc[
        n_train
        + n_validation:
    ]["id_contrato"]
)

split_map = {
    **{
        cid: "train"
        for cid in ids_train
    },
    **{
        cid: "validation"
        for cid in ids_validation
    },
    **{
        cid: "test"
        for cid in ids_test
    },
}

dataset[
    "split_contrato_temporal"
] = (
    dataset[
        "id_contrato"
    ].map(
        split_map
    )
)

dataset[
    "split_modelagem"
] = np.where(
    dataset[
        "flag_apto_treino"
    ].eq(1),
    dataset[
        "split_contrato_temporal"
    ],
    "nao_supervisionado",
)

dataset = dataset.sort_values(
    [
        "id_contrato",
        "data_corte",
    ]
).reset_index(drop=True)

split_summary = (
    dataset[
        dataset[
            "flag_apto_treino"
        ].eq(1)
    ]
    .groupby(
        "split_modelagem"
    )
    .agg(
        snapshots=(
            "id_contrato",
            "size",
        ),
        contratos=(
            "id_contrato",
            "nunique",
        ),
        taxa_atraso=(
            "target_atraso",
            "mean",
        ),
        mediana_atraso_dias=(
            "target_dias_atraso",
            "median",
        ),
    )
)

display(split_summary)


,snapshots,contratos,taxa_atraso,mediana_atraso_dias
split_modelagem,,,,
test,149,8,0.1745,0.0000
train,372,33,0.5134,12.0000
validation,64,7,0.5312,14.0000


## 11. Validações finais contra inconsistência e leakage

In [12]:

# Universo.
assert (
    set(
        dataset[
            "id_contrato"
        ].unique()
    )
    == ids_modelagem
)

# Unicidade.
assert not dataset.duplicated(
    [
        "id_contrato",
        "data_corte",
    ]
).any()

# Snapshot supervisionado anterior à conclusão.
train_mask = dataset[
    "flag_apto_treino"
].eq(1)

assert (
    dataset.loc[
        train_mask,
        "data_corte",
    ]
    <
    dataset.loc[
        train_mask,
        "data_conclusao_definitiva",
    ]
).all()

# Classificação pré-prazo.
clf_mask = dataset[
    "flag_apto_classificacao"
].eq(1)

assert (
    dataset.loc[
        clf_mask,
        "dias_restantes_prazo_vigente",
    ].ge(0)
).all()

# Target binário.
assert np.array_equal(
    (
        dataset[
            "target_dias_atraso"
        ] > 0
    ).astype(int).to_numpy(),
    dataset[
        "target_atraso"
    ].astype(int).to_numpy(),
)

# Target em dias.
target_recalc = (
    dataset[
        "data_conclusao_definitiva"
    ]
    - dataset[
        "data_termino_vigente_final_controle"
    ]
).dt.days.clip(
    lower=0
)

assert np.allclose(
    target_recalc.astype(float),
    dataset[
        "target_dias_atraso"
    ].astype(float),
)

# Contrato em um único split.
assert (
    dataset.groupby(
        "id_contrato"
    )[
        "split_contrato_temporal"
    ]
    .nunique()
    .max()
    == 1
)

# Peso de regressão igual por contrato.
reg_weight_totals = (
    dataset[
        dataset[
            "flag_apto_treino"
        ].eq(1)
    ]
    .groupby(
        "id_contrato"
    )[
        "sample_weight_contrato"
    ]
    .sum()
)

assert np.allclose(
    reg_weight_totals.values,
    reg_weight_totals.iloc[0],
)

# Peso de classificação igual por contrato no subconjunto pré-prazo.
clf_weight_totals = (
    dataset[
        dataset[
            "flag_apto_classificacao"
        ].eq(1)
    ]
    .groupby(
        "id_contrato"
    )[
        "sample_weight_classificacao"
    ]
    .sum()
)

assert np.allclose(
    clf_weight_totals.values,
    clf_weight_totals.iloc[0],
)

print(
    "Todas as validações foram concluídas com sucesso."
)


Todas as validações foram concluídas com sucesso.



## 12. Dicionário de dados e proteção contra leakage

Campos finais, IDs de alta cardinalidade, pesos e splits são classificados como controle.

O notebook 03 deve selecionar apenas colunas marcadas como `feature_candidata`.


In [13]:

CONTROL_COLUMNS = {
    "id_contrato",
    "data_corte",

    "data_os",
    "data_termino_original",
    "data_termino_vigente_corte",

    "status_final_controle",
    "data_conclusao_sistema_controle",
    "data_conclusao_definitiva",
    "data_conclusao_controle",
    "data_termino_vigente_final_controle",
    "diferenca_vigente_original_final_controle",

    "target_dias_atraso",
    "target_atraso",

    "flag_apto_treino",
    "flag_apto_classificacao",

    "sample_weight_contrato",
    "sample_weight_classificacao",

    "split_contrato_temporal",
    "split_modelagem",
}

TARGET_COLUMNS = {
    "target_dias_atraso",
    "target_atraso",
}

dictionary_rows = []

for column in dataset.columns:
    if column in TARGET_COLUMNS:
        role = "target"
    elif column in CONTROL_COLUMNS:
        role = "controle"
    else:
        role = "feature_candidata"

    dictionary_rows.append({
        "coluna": column,
        "papel": role,
        "dtype": str(
            dataset[column].dtype
        ),
        "nulos_pct": round(
            dataset[
                column
            ].isna().mean()
            * 100,
            2,
        ),
        "nunique": (
            dataset[
                column
            ].nunique(
                dropna=True
            )
        ),
    })

dictionary = pd.DataFrame(
    dictionary_rows
)

display(dictionary)


,coluna,papel,dtype,nulos_pct,nunique
0,id_contrato,controle,int64,0.0000,48
1,data_corte,controle,datetime64[us],0.0000,189
2,classificacao_contrato,feature_candidata,str,0.0000,3
3,id_setor_divisao,feature_candidata,int64,0.0000,4
4,ano_contrato,feature_candidata,int64,0.0000,8
...,...,...,...,...,...
104,flag_apto_classificacao,controle,int8,0.0000,2
105,sample_weight_contrato,controle,float64,0.0000,19
106,sample_weight_classificacao,controle,float64,0.0000,19
107,split_contrato_temporal,controle,str,0.0000,3


## 13. Exportação

In [14]:

OUTPUT_ENRIQUECIDO = (
    PROCESSED
    / "dataset_modelagem_obras_snapshots_enriquecido.csv"
)

OUTPUT_CANONICO = (
    PROCESSED
    / "dataset_modelagem_obras_snapshots.csv"
)

DICTIONARY_PATH = (
    PROCESSED
    / "dicionario_dataset_modelagem_enriquecido.csv"
)

QUALITY_PATH = (
    QUALITY
    / "qualidade_dataset_modelagem_enriquecido.csv"
)

dataset.to_csv(
    OUTPUT_ENRIQUECIDO,
    index=False,
    encoding="utf-8-sig",
)

dataset.to_csv(
    OUTPUT_CANONICO,
    index=False,
    encoding="utf-8-sig",
)

dictionary.to_csv(
    DICTIONARY_PATH,
    index=False,
    encoding="utf-8-sig",
)

quality = pd.DataFrame({
    "metrica": [
        "contratos",
        "contratos_com_atraso",
        "contratos_sem_atraso",
        "taxa_atraso_contratos",
        "snapshots",
        "snapshots_aptos_treino",
        "snapshots_aptos_classificacao",
        "contratos_aptos_treino",
        "colunas_dataset",
        "features_candidatas",
        "data_conclusao_min",
        "data_conclusao_max",
        "target_dias_media",
        "target_dias_mediana",
        "target_dias_max",
        "contratos_revisao_semantica",
    ],
    "valor": [
        contratos[
            "idContrato"
        ].nunique(),
        int(
            contratos[
                "target_atraso"
            ].sum()
        ),
        int(
            (
                contratos[
                    "target_atraso"
                ] == 0
            ).sum()
        ),
        float(
            contratos[
                "target_atraso"
            ].mean()
        ),
        len(dataset),
        int(
            dataset[
                "flag_apto_treino"
            ].sum()
        ),
        int(
            dataset[
                "flag_apto_classificacao"
            ].sum()
        ),
        dataset.loc[
            dataset[
                "flag_apto_treino"
            ].eq(1),
            "id_contrato",
        ].nunique(),
        dataset.shape[1],
        int(
            dictionary[
                "papel"
            ].eq(
                "feature_candidata"
            ).sum()
        ),
        contratos[
            "data_recebimento_provisorio"
        ].min(),
        contratos[
            "data_recebimento_provisorio"
        ].max(),
        float(
            contratos[
                "target_dias_atraso"
            ].mean()
        ),
        float(
            contratos[
                "target_dias_atraso"
            ].median()
        ),
        float(
            contratos[
                "target_dias_atraso"
            ].max()
        ),
        len(target_review),
    ],
})

quality.to_csv(
    QUALITY_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("=" * 72)
print("DATASET ENRIQUECIDO GERADO")
print("=" * 72)
print("Dataset enriquecido:")
print(OUTPUT_ENRIQUECIDO.resolve())
print()
print("Dataset canônico:")
print(OUTPUT_CANONICO.resolve())
print()
print("Dicionário:")
print(DICTIONARY_PATH.resolve())
print()
print("Relatório de qualidade:")
print(QUALITY_PATH.resolve())
print()
display(quality)


DATASET ENRIQUECIDO GERADO
Dataset enriquecido:
C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\data\processed\dataset_modelagem_obras_snapshots_enriquecido.csv

Dataset canônico:
C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\data\processed\dataset_modelagem_obras_snapshots.csv

Dicionário:
C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\data\processed\dicionario_dataset_modelagem_enriquecido.csv

Relatório de qualidade:
C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\reports\quality\qualidade_dataset_modelagem_enriquecido.csv



,metrica,valor
0,contratos,48
1,contratos_com_atraso,20
2,contratos_sem_atraso,28
3,taxa_atraso_contratos,0.4167
4,snapshots,633
5,snapshots_aptos_treino,585
6,snapshots_aptos_classificacao,509
7,contratos_aptos_treino,48
8,colunas_dataset,109
9,features_candidatas,90
